# EDA Bronze — exploración de datos crudos

Análisis exploratorio de los **2 datasets que ingesta la capa bronze** (`pipeline/assets/bronze.py`)
para diseñar las transformaciones de **silver/gold** y los **chequeos de calidad** (ADRs 0013-0015).

- **Motor:** DuckDB (SQL-first, alineado con dbt) + pandas para resúmenes.
- **Fuente:** CSV locales en `../data/` (mismos bytes que el bronze; sin credenciales).
- **Entorno:** conda `eda-bronze` (`environment.yml`). NO usa el poetry de producción.
- **Out of scope:** `produccin-...-2026.csv` (convencional) NO es parte del bronze.

Los hallazgos destilados viven en `../docs/data/eda-bronze.md`.

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

DATA = '../data'
LISTADO = f'{DATA}/listado-de-pozos-cargados-por-empresas-operadoras.csv'
PROD = f'{DATA}/produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv'

con = duckdb.connect()
# normalize_names + el lector de DuckDB limpian el BOM del header (utf-8-sig).
con.execute(f"CREATE VIEW listado AS SELECT * FROM read_csv_auto('{LISTADO}', normalize_names=true)")
con.execute(f"CREATE VIEW prod AS SELECT * FROM read_csv_auto('{PROD}', normalize_names=true)")
print('Vistas creadas: listado, prod')

Vistas creadas: listado, prod


## 1. listado_pozos — schema, grano y nulos

In [2]:
print('shape:', con.sql('SELECT count(*) FROM listado').fetchone()[0], 'filas')
con.sql('DESCRIBE listado').df()

shape: 84242 filas


,column_name,column_type,null,key,default,extra
0,idpozo,BIGINT,YES,None,None,None
1,sigla,VARCHAR,YES,None,None,None
2,formprod,VARCHAR,YES,None,None,None
3,idempresa,VARCHAR,YES,None,None,None
4,idareapermisoconcesion,VARCHAR,YES,None,None,None
5,idareayacimiento,VARCHAR,YES,None,None,None
6,idcuenca,VARCHAR,YES,None,None,None
7,idprovincia,VARCHAR,YES,None,None,None
8,codigopropio,VARCHAR,YES,None,None,None
9,nombrepropio,VARCHAR,YES,None,None,None


In [3]:
# Grano: idpozo unico, sin nulos, sin duplicados?
con.sql('''
    SELECT count(*) AS filas,
           count(DISTINCT idpozo) AS idpozo_distintos,
           count(*) FILTER (WHERE idpozo IS NULL) AS idpozo_nulos
    FROM listado
''').df()

,filas,idpozo_distintos,idpozo_nulos
0,84242,84242,0


In [4]:
# % de nulos por columna (top 20)
lp = con.sql('SELECT * FROM listado').df()
(lp.isna().mean().mul(100).round(1).sort_values(ascending=False).head(20)
   .rename('pct_nulos').to_frame())

,pct_nulos
adjiv_fecha_abandono,95.1
subtipo_reservorio,94.5
adjiv_subtipo_reservorio,94.4
fechadeingreso,81.6
fecha_data,81.6
adjiv_fecha_fin_term,42.6
adjiv_fecha_inicio_term,42.6
adjiv_capacidad_perf,40.3
adjiv_fecha_fin,39.9
comp_perf,39.9


In [5]:
# Categoricas candidatas a dimension
for col in ['cuenca', 'provincia', 'tipo_reservorio', 'clasificacion', 'gasplus']:
    print(f'\n== {col} ==')
    print(con.sql(f'SELECT {col}, count(*) AS n FROM listado GROUP BY 1 ORDER BY n DESC LIMIT 8').df().to_string(index=False))


== cuenca ==
         cuenca     n
GOLFO SAN JORGE 44045
       NEUQUINA 32366
         CUYANA  3684
        AUSTRAL  3133
       NOROESTE   996
        NORESTE    11
            NaN     3
       ÑIRIHUAU     2

== provincia ==
       provincia     n
      Santa Cruz 23580
          Chubut 22298
         Neuquén 18950
         Mendoza  8744
       Rio Negro  5646
        La Pampa  2710
Tierra del Fuego  1242
           Salta   896

== tipo_reservorio ==
tipo_reservorio     n
   CONVENCIONAL 57690
            NaN 21518
NO CONVENCIONAL  4633
 SIN RESERVORIO   396
NO DISCRIMINADO     5

== clasificacion ==
 clasificacion     n
   EXPLOTACION 56851
           NaN 17939
   EXPLORACION  5432
      SERVICIO  3987
ALMACENAMIENTO    33

== gasplus ==
gasplus     n
     no 83136
     si  1106


In [6]:
# Outliers numericos (profundidad, coordenadas)
con.sql('''
    SELECT
        min(profundidad) AS prof_min, max(profundidad) AS prof_max, avg(profundidad) AS prof_avg,
        min(coordenadax) AS x_min, max(coordenadax) AS x_max,
        min(coordenaday) AS y_min, max(coordenaday) AS y_max
    FROM listado
''').df()

,prof_min,prof_max,prof_avg,x_min,x_max,y_min,y_max
0,0.0,378939.0,1698.195063,-72.1059,55.37631,-69.41577,-22.00255


## 2. produccion_no_convencional — grano y duplicados

In [7]:
print('shape:', con.sql('SELECT count(*) FROM prod').fetchone()[0], 'filas')
# Grano (idpozo, anio, mes): duplicados?
con.sql('''
    SELECT count(*) AS filas,
           count(DISTINCT (idpozo, anio, mes)) AS grano_distinto,
           count(*) - count(DISTINCT (idpozo, anio, mes)) AS duplicados_grano,
           min(anio) AS anio_min, max(anio) AS anio_max
    FROM prod
''').df()

shape: 405996 filas


,filas,grano_distinto,duplicados_grano,anio_min,anio_max
0,405996,405996,0,2006,2026


In [8]:
# rectificado: flag de correccion (clave para el tipo de carga, ADR-0013)
con.sql('SELECT rectificado, count(*) AS n FROM prod GROUP BY 1 ORDER BY n DESC').df()

,rectificado,n
0,False,405302
1,True,694


In [9]:
# Columnas constantes / casi vacias -> no promover a gold
pr = con.sql('SELECT * FROM prod').df()
print('habilitado:'); print(pr['habilitado'].value_counts(dropna=False))
print('\ntipo_de_recurso:'); print(pr['tipo_de_recurso'].value_counts(dropna=False))
print('\n% nulos (top 10):')
print(pr.isna().mean().mul(100).round(1).sort_values(ascending=False).head(10))

habilitado:
habilitado
True    405996
Name: count, dtype: int64

tipo_de_recurso:
tipo_de_recurso
NO CONVENCIONAL    405996
Name: count, dtype: int64

% nulos (top 10):
vida_util           97.9
observaciones       94.3
clasificacion        0.2
subclasificacion     0.2
tipopozo             0.1
tipoextraccion       0.1
sub_tipo_recurso     0.1
tipoestado           0.1
iny_agua             0.0
prod_agua            0.0
dtype: float64


In [10]:
# Categoricas: estado, tipo de pozo, extraccion, recurso
for col in ['tipoestado', 'tipopozo', 'tipoextraccion', 'sub_tipo_recurso']:
    print(f'\n== {col} ==')
    print(con.sql(f'SELECT {col}, count(*) AS n FROM prod GROUP BY 1 ORDER BY n DESC LIMIT 8').df().to_string(index=False))


== tipoestado ==
              tipoestado      n
     Extracción Efectiva 335603
 Parado Transitoriamente  25484
              En Estudio  23206
              Abandonado   3521
             A Abandonar   3483
Otras Situación Inactivo   3198
       En Reserva de Gas   2966
 En Espera de Reparación   2583

== tipopozo ==
         tipopozo      n
         Gasífero 221257
      Petrolífero 156071
        Otro tipo  27320
         Sumidero    644
              NaN    605
Inyección de Agua     56
 Inyección de Gas     43

== tipoextraccion ==
           tipoextraccion      n
        Surgencia Natural 261982
             Plunger Lift  66789
          Bombeo Mecánico  44073
Sin Sistema de Extracción  17747
                 Gas Lift  13996
                      NaN    605
        Bombeo Hidráulico    336
        Electrosumergible    257

== sub_tipo_recurso ==
sub_tipo_recurso      n
           SHALE 213916
           TIGHT 191640
             NaN    440


In [11]:
# Medidas de la fact: rangos, ceros y negativos (errores de fuente)
con.sql('''
    SELECT 'prod_pet' AS medida, min(prod_pet) AS mn, max(prod_pet) AS mx, avg(prod_pet) AS avg,
           count(*) FILTER (WHERE prod_pet < 0) AS negativos, count(*) FILTER (WHERE prod_pet = 0) AS ceros FROM prod
    UNION ALL SELECT 'prod_gas', min(prod_gas), max(prod_gas), avg(prod_gas),
           count(*) FILTER (WHERE prod_gas < 0), count(*) FILTER (WHERE prod_gas = 0) FROM prod
    UNION ALL SELECT 'prod_agua', min(prod_agua), max(prod_agua), avg(prod_agua),
           count(*) FILTER (WHERE prod_agua < 0), count(*) FILTER (WHERE prod_agua = 0) FROM prod
    UNION ALL SELECT 'profundidad', min(profundidad), max(profundidad), avg(profundidad),
           count(*) FILTER (WHERE profundidad < 0), count(*) FILTER (WHERE profundidad = 0) FROM prod
''').df()

,medida,mn,mx,avg,negativos,ceros
0,prod_pet,-0.001,26593.261,323.080680,1,144047
1,prod_gas,-12.267,29129.660,628.028068,2,83800
2,prod_agua,0.000,34792.950,182.208254,0,122426
3,profundidad,0.000,378939.000,3743.856595,0,5275


## 3. Integridad referencial produccion.idpozo → listado.idpozo

In [12]:
# Pozos en produccion sin match en listado (huerfanos) -> test relationships en warn
con.sql('''
    WITH huerfanos AS (
        SELECT DISTINCT p.idpozo
        FROM prod p LEFT JOIN listado l ON p.idpozo = l.idpozo
        WHERE l.idpozo IS NULL
    )
    SELECT (SELECT count(DISTINCT idpozo) FROM prod) AS pozos_en_prod,
           (SELECT count(DISTINCT idpozo) FROM listado) AS pozos_en_listado,
           (SELECT count(*) FROM huerfanos) AS pozos_huerfanos,
           (SELECT count(*) FROM prod WHERE idpozo IN (SELECT idpozo FROM huerfanos)) AS filas_huerfanas
''').df()

,pozos_en_prod,pozos_en_listado,pozos_huerfanos,filas_huerfanas
0,4929,84242,296,2704


## 4. Cross-check con el diccionario oficial (`../docs/data/dataset_readme.md`)

El readme de la Secretaría de Energía aporta **unidades** (gas en miles de m³, resto en m³) y
revela que las **coordenadas están etiquetadas al revés**. Lo verificamos contra los datos:
`coordenaday` sigue el gradiente norte→sur de Argentina (es **latitud**) y `coordenadax` queda
en el rango de **longitud** (~−68). También chequeamos `tef ≤ 31` días y `fecha_data` = fin de mes.

In [13]:
# El readme dice coordenadax=Latitud, coordenaday=Longitud. Verificamos por provincia:
# coordenaday sigue el gradiente norte->sur (Salta -24 ... Santa Cruz -50) => es LATITUD.
con.sql('''
    SELECT provincia,
           round(avg(coordenadax), 2) AS x_avg_es_longitud,
           round(avg(coordenaday), 2) AS y_avg_es_latitud,
           count(*) AS n
    FROM prod GROUP BY 1 ORDER BY y_avg_es_latitud DESC
''').df()

,provincia,x_avg_es_longitud,y_avg_es_latitud,n
0,Salta,-63.99,-24.12,185
1,Mendoza,-69.66,-35.64,3057
2,Neuquén,-68.83,-38.47,357889
3,Rio Negro,-67.83,-38.97,28713
4,Chubut,-67.85,-45.83,253
5,Santa Cruz,-70.74,-50.65,15899


In [14]:
# Rangos: tef debe ser <= 31 dias; coordenadas dentro de Argentina (lat [-56,-21], lon [-74,-53])
con.sql('''
    SELECT max(tef) AS tef_max,
           round(quantile_cont(tef, 0.99), 1) AS tef_p99,
           count(*) FILTER (WHERE tef > 31) AS tef_fuera_rango,
           count(*) FILTER (WHERE coordenaday < -56 OR coordenaday > -21) AS lat_fuera_rango,
           count(*) FILTER (WHERE coordenadax < -74 OR coordenadax > -53) AS lon_fuera_rango
    FROM prod
''').df()

,tef_max,tef_p99,tef_fuera_rango,lat_fuera_rango,lon_fuera_rango
0,79.34,31.0,15,65,65


In [15]:
# fecha_data = ultimo dia del mes del periodo (anio, mes); respeta bisiestos -> date key de la fact
con.sql('''
    SELECT DISTINCT anio, mes, fecha_data
    FROM prod
    WHERE (anio = 2024 AND mes = 2)   -- bisiesto: esperado 2024-02-29
       OR (anio = 2026 AND mes = 1)
       OR (anio = 2025 AND mes = 11)
    ORDER BY anio, mes
''').df()

,anio,mes,fecha_data
0,2024,2,2024-02-29
1,2025,11,2025-11-30
2,2026,1,2026-01-31


## Conclusiones

Ver el detalle y el mapeo a transformaciones silver, modelo estrella y tests de calidad en
[`../docs/data/eda-bronze.md`](../docs/data/eda-bronze.md).